# 2018–2022韩国海关前20日半导体出口原报同比：数值来源

固定HWP原始字节出口段解析与预览交叉核对，不读取SOXX行情或建立交易信号。


In [ ]:
import hashlib,json,os,pathlib,re,zlib
import numpy as np,pandas as pd,qlib,olefile
from qlib.workflow import R
from src.data.nas_market_data import archive_nas_recorder,_settings,_ssh
import shlex
root=pathlib.Path.cwd();card=root/'research/kcs_early20_2018_2022_semiconductor_yoy_source_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];sha=lambda p:hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest();assert plan['frozen'] and not (card/'run_state.json').exists() and olefile.__version__=='0.47'
capture=root/src['capture_card'];snap=root/src['snapshot_root'];assert sha(capture/'summary.json')==src['capture_summary_sha256'] and sha(capture/'source_quality.json')==src['source_quality_sha256']
source_summary=json.loads((capture/'summary.json').read_text());assert json.loads((capture/'source_restore.json').read_text())['status']=='NAS_five_year_original_groups_all_members_cold_restored_exact' and len(source_summary['year_archives'])==5
for year,h in src['year_manifest_sha256'].items():assert sha(snap/year/'year_manifest.json')==h
pat=re.compile(r'^반도체\s*\(\s*([△▽▲▼−\-+]?\s*[\d,.]+)\s*%\s*\)');preview_pat=re.compile(r'반도체\s*\(\s*([△▽▲▼−\-+]?\s*[\d,.]+)\s*%\s*\)');label='반도체'.encode('utf-16le');export='수출현황'.encode('utf-16le');imported='수입현황'.encode('utf-16le')
def number(token):
 text=re.sub(r'\s+','',token);negative=text[:1] in ('△','▽','▼','−','-');text=text.lstrip('△▽▲▼−-+').replace(',','');value=float(text);assert np.isfinite(value);return -value if negative else value
state={'status':'HWP_numeric_reads_started','plan_sha256':sha(card/'plan.json'),'HWP_reads':0,'preview_corrob':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');rows=[];details=[]
for year in range(2018,2023):
 manifest=json.loads((snap/str(year)/'year_manifest.json').read_text());assert len(manifest['months'])==12;year_archive=next(x for x in source_summary['year_archives'] if x['year']==year)
 for month in manifest['months']:
  hwps=[x for x in month['attachments'] if x['ext']=='hwp'];assert len(hwps)==1 and hwps[0]['signature_ok'];a=hwps[0];path=root/a['local_path'];assert sha(path)==a['body_sha256'] and path.stat().st_size==a['body_bytes'] and olefile.isOleFile(str(path));state['HWP_reads']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
  with olefile.OleFileIO(str(path)) as doc:
   assert doc.exists('BodyText/Section0') and doc.exists('PrvText');body=zlib.decompress(doc.openstream('BodyText/Section0').read(),-15);preview=doc.openstream('PrvText').read().decode('utf-16le','replace');meta=doc.get_metadata();created=str(meta.create_time) if meta.create_time else None;saved=str(meta.last_saved_time) if meta.last_saved_time else None
  start=body.find(export);end=body.find(imported,start+len(export)) if start>=0 else -1;assert start>=0 and (end<0 or end>start);candidates=[];at=0
  while True:
   at=body.find(label,at)
   if at<0:break
   if at>start and (end<0 or at<end):
    snippet=body[at:at+130].decode('utf-16le','replace');match=pat.search(snippet)
    if match:candidates.append({'label_byte_offset':at,'token':match.group(1),'value':number(match.group(1))})
   at+=len(label)
  assert len(candidates)==1,(month['month'],candidates);found=candidates[0];ps=preview.find('수출현황');pe=preview.find('수입현황',ps+4) if ps>=0 else -1;assert ps>=0;ptext=preview[ps:pe if pe>=0 else None];pmatches=preview_pat.findall(ptext)
  if month['month']=='2022-09':assert len(pmatches)==0 and len(preview)<=1023
  else:
   assert len(pmatches)==1 and abs(number(pmatches[0])-found['value'])<1e-9,(month['month'],pmatches,found);state['preview_corrob']+=1
  rows.append({'period':month['month'],'reported_semiconductor_export_yoy_pct':found['value'],'raw_token':found['token'],'HWP_sha256':a['body_sha256'],'HWP_bytes':a['body_bytes'],'NAS_year_archive_sha256':year_archive['sha256'],'preview_corroborated':len(pmatches)==1,'HWP_created_metadata':created,'HWP_last_saved_metadata':saved})
  details.append({'period':month['month'],'export_heading_byte_offset':start,'import_heading_byte_offset':end,'match':found,'preview_export_char_offset':ps,'preview_import_char_offset':pe,'preview_matches':pmatches,'body_decompressed_bytes':len(body),'preview_chars':len(preview),'HWP_file_path':a['local_path'],'HWP_sha256':a['body_sha256']})
assert state['HWP_reads']==60 and state['preview_corrob']==59 and len(rows)==len(details)==60;table=pd.DataFrame(rows);assert table.period.tolist()==[f'{y}-{m:02d}' for y in range(2018,2023) for m in range(1,13)]
anchors={'2018-10':9.4,'2019-10':-28.8,'2020-10':12.1,'2021-10':23.9,'2022-10':-12.8};assert all(abs(float(table.loc[table.period.eq(k),'reported_semiconductor_export_yoy_pct'].iloc[0])-v)<1e-9 for k,v in anchors.items())
result={'status':'official_2018_2022_KCS_early20_reported_semiconductor_export_YoY_numeric_source_complete_PIT_pending','plan_sha256':sha(card/'plan.json'),'monthly_rows':60,'HWP_source_reads':60,'single_export_prose_match_months':60,'preview_corrob_months':59,'preview_truncated_unknown':['2022-09'],'negative_YoY_months':int(table.reported_semiconductor_export_yoy_pct.lt(0).sum()),'YoY_min':float(table.reported_semiconductor_export_yoy_pct.min()),'YoY_max':float(table.reported_semiconductor_export_yoy_pct.max()),'October_original_anchor_values':anchors,'source_first_public_vintage_qualified':False,'working_day_adjusted':False,'held_2023_2026_source_available':False,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/'data/qlib/minute_information_pit_allocator_20261012/daily_prices'),region='us',kernels=1)
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,source_archive_ids=[x['sha256'] for x in source_summary['year_archives']]);rec.save_objects(artifact_path='analysis',monthly_reported_YoY=table,source_offsets_and_preview=details,result=result);rec.log_metrics(source_months=60,preview_corroborated=59,source_only=int(True),model_fits=0,native_accounts=0);rid,eid=rec.id,str(rec.experiment_id)
state['status']='finished';state['recorder_id']=rid;state['experiment_id']=eid;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid);assert _ssh(_settings(root)['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0]==evidence['sha256']
summary={'experiment':plan['experiment'],'decision':'KCS_2018_2022_early20_reported_semiconductor_YoY_source_numeric_complete_held_and_PIT_pending','objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','recorder_id':rid,'experiment_id':eid,'recorder_uri':str(card/'qlib_recorder'),'NAS_evidence':evidence,'NAS_readback':{'sha256':evidence['sha256'],'verified':True},'monthly_rows':60,'preview_corroborated':59,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/monthly_reported_YoY','analysis/source_offsets_and_preview','analysis/result']},'limitations':['Reported raw YoY is working-day sensitive and not a demand surprise.','Current official HWP bytes and metadata do not certify historic first-public versions.','No 2023-2026 held numeric panel, US-session Qlib factor, SOXX result or strategy account.']};(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n');print(json.dumps({'decision':summary['decision'],'monthly_rows':60,'preview_corrob':59,'negative':result['negative_YoY_months'],'min':result['YoY_min'],'max':result['YoY_max'],'recorder_id':rid,'NAS_sha256':evidence['sha256']},ensure_ascii=False,indent=2))


## 中文结论

对NAS已逐件冷恢复的60份官方HWP，固定在原始BodyText的“出口现状”至“进口现状”之间定位半导体产品同比行，**60/60月各且仅有一个数值**；HWP预览另对59个月逐值一致，2022-09预览在该行之前截断，正文中仍唯一可定位。原始负号 `△` 保留语义转成负值，未使用进口半导体行或表头合计数字。已核对的10月锚点2018–2022分别为 **+9.4%、-28.8%、+12.1%、+23.9%、-12.8%**，与此前2021/2022原稿核对一致。60个月中负同比 **19** 个月，范围 **-39.8% 至 +49.5%**。

完整60行数值、原件SHA、正文/预览偏移、元数据保存在原生Qlib Recorder `e700f88dcb824ab88c4d1d0e13fc3fa0`；NAS SHA-256 `0b7dfd3a7eb4da857a9efcdcb868a9c1a8d5090202013a5d13aba65fb1ba54cb` 的数值表、逐份定位与结果工件已限定冷恢复一致。原件字节本身已按五个年度分别存NAS，故本卡未重复下载60份文件。

决定 `KCS_2018_2022_early20_reported_semiconductor_YoY_source_numeric_complete_held_and_PIT_pending`。该数值是**未经韩国工作日差异调整的暂定同比**，并非需求意外；当前可下载HWP、门户登记日期及内部元数据不足以认证历史首次公开版本。更关键的是，2023–2026同口径月度原件尚未构成留出面，不能把这60个训练月填入Stage1后直接声称独立信息增量。下一步需另冻留出来源抓取、逐月版本与决策时钟审核，再按原Stage1信息和0bp完整账户硬门检验。本卡0 SOXX收益读取、0拟合、0账户，目标未达成。
